# Final Project: Deep Fake Detection

by Patrick Donnelly & Burke Havranek

EECE 5644: Introduction to Machine Learning and Pattern Recognition

Northeastern University College of Engineering

Summer 2026, Session B

## Part 1: Loading and Sanitization

### Required Imports:

In [ ]:
#!/bin/python3
# --Beginning of Code--
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "pandas", "kagglehub", "ipywidgets", "parfor", "torch", "torchvision")

# Standard processing
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 80)
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

# Useful built-ins
import shutil
from parfor import parfor
from PIL import Image

# Anticipated framework
import torch
from torchvision.transforms.functional import to_tensor

# Constants
IMAGE_SIZE = 224 # Common CV standard
TRAIN_FRAC = 0.8 # Common industry standard
SEED = 0xDEADBEEF # Humorous personal standard
DOWNSAMPLER = Image.LANCZOS # Industry standard downsampler

RAW_DIR = "images" # Determined by Kaggle
PRE_DIR = "preprocessed-images" # Arbitrary

REAL_DIR = "real" # Determined by Kaggle
FAKE_DIR = "fake" # Determined by Kaggle
FLUX_DIR = ["FLUX_DEV", "FLUX_PRO", "SDXL"] # Determined by Kaggle

N_REAL = 70_000
N_FAKE = [7273, 3209, 53087]

# ImageNet normalization
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

### Download and Verify Raw Data:

In [ ]:
# Check whether raw data is loaded
if not os.path.isdir(f"{RAW_DIR}"):
    print("RAW DATA DIRECTORY NOT FOUND, DOWNLOADING...", end="")
    
    # Download data
    path = kagglehub.dataset_download("shreyanshpatel1/130k-real-vs-fake-face")
    root = os.path.join(path, f"{RAW_DIR}")
    
    print(f"DONE\nCOPYING DATA FROM {root}...", end="")
    
    # Copy to local directory structure
    shutil.copytree(root, f"./{RAW_DIR}")
    
    print("DONE\nSETTING PERMISSIONS...", end="")
    
    # Change raw data to read-only for safety (parallelized)
    for root, _, files in os.walk(f"./{RAW_DIR}"):
        @parfor(files, (root,))
        def make_read_only(file, root):
            os.chmod(os.path.join(root, file), 0o444)
    
    print("DONE")
else:
    print("RAW DATA DIRECTORY FOUND, CONTINUING")

print("VERIFYING RAW DATA INTEGRITY...", end="")

try:
    # Verify raw data integrity
    assert len(os.listdir(f"./{RAW_DIR}")) == 2
    assert len(os.listdir(f"./{RAW_DIR}/{REAL_DIR}")) == N_REAL
    assert len(os.listdir(f"./{RAW_DIR}/{FAKE_DIR}")) == len(FLUX_DIR)
    for d, N in zip(FLUX_DIR, N_FAKE):
        assert len(os.listdir(f"./{RAW_DIR}/{FAKE_DIR}/{d}")) == N
except:
    raise RuntimeError("RAW DATA INTEGRITY COMPROMISED, PLEASE REMOVE AND RE-DOWNLOAD")

print("DONE\nRAW DATA SUCCESSFULLY VERIFIED")

### Create Directory Structure for Sanitized Data:

In [ ]:
if not os.path.isdir(f"{PRE_DIR}"):
    print("SANITIZED DATA DIRECTORY NOT FOUND, CREATING STRUCTURE...", end="")
    
    os.mkdir(f"{PRE_DIR}")
    
    # Copy directories, but not files
    for root, dirs, _ in os.walk(f"./{RAW_DIR}"):
        sanitized_root = root.replace(RAW_DIR, PRE_DIR)
        for dir in dirs:
            os.mkdir(os.path.join(sanitized_root, dir))
            
    print("DONE")
else:
    print("SANITIZED DATA DIRECTORY FOUND, CONTINUING")

print("VERIFYING SANITIZED DATA DIRECTORY STRUCTURE...", end="")

try:
    # Verify sanitized data directory structure integrity
    assert len(os.listdir(f"./{PRE_DIR}")) == 2
    assert len(os.listdir(f"./{PRE_DIR}/{FAKE_DIR}")) == len(FLUX_DIR)
except:
    raise RuntimeError("SANITIZED DATA DIRECTORY STRUCTURE COMPROMISED, PLEASE REMOVE AND RE-RUN")

print("DONE\nSANITIZED DATA DIRECTORY STRUCTURE SUCCESSFULLY VERIFIED")

### Preprocess Images:
- Load all images and verify color encoding (RGB)
- Downsample all images to same resolution using the Lanczos algorithm
- Encode and normalize all images using PyTorch vector
- Save to same directory structure

In [ ]:
# Iterate over each image and preprocess (parallelized)
if os.path.isdir(f"{PRE_DIR}") and len(os.listdir(f"./{PRE_DIR}/{REAL_DIR}")) == 0:
    print("PREPROCESSING IMAGE DATA...", end="")
    
    for root, _, files in os.walk(f"./{RAW_DIR}"):
        @parfor(files, (root, RAW_DIR, PRE_DIR, IMAGE_SIZE, DOWNSAMPLER, MEAN, STD))
        def preprocess_image(file, root, raw_dir, san_dir, res, alg, u, o):
            raw_path = os.path.join(root, file)
            san_path = os.path.splitext(raw_path.replace(raw_dir, san_dir))[0] + ".pt"
            
            # Load image, verify encoding, resize using Lanczos
            img = Image.open(raw_path).convert("RGB").resize((res, res), resample=alg)
            
            # Convert to PyTorch, then normalize using ImageNet standard
            ten = (to_tensor(img).float() - u) / o
            
            # Export to new directory
            torch.save(ten, san_path)
            
    print("DONE")
else:
    print("SANITIZED DATA FOUND, CONTINUING")

print("VERIFYING SANITIZED DATA INTEGRITY...", end="")

try:
    # Verify sanitized data integrity
    assert len(os.listdir(f"./{PRE_DIR}")) == 2
    assert len(os.listdir(f"./{PRE_DIR}/{REAL_DIR}")) == N_REAL
    assert len(os.listdir(f"./{PRE_DIR}/{FAKE_DIR}")) == len(FLUX_DIR)
    for d, N in zip(FLUX_DIR, N_FAKE):
        assert len(os.listdir(f"./{PRE_DIR}/{FAKE_DIR}/{d}")) == N
except:
    raise RuntimeError("SANITIZED DATA INTEGRITY COMPROMISED, PLEASE REMOVE AND RE_PROCESS")

print("DONE\nSANITIZED DATA SUCCESSFULLY VERIFIED")

### Organize Data:

Refactor structure into level directory with associated `index.csv` for data analysis:

In [ ]:
N = N_REAL + sum(N_FAKE)
idx = np.zeros((N,), dtype="<U10")
key = np.zeros((4,N), dtype=np.uint8)

# Iterative due to renaming convention
i = 0
if not os.path.exists(f"{PRE_DIR}/index.csv"):
    print("FINALIZING IMAGE DATA...", end="")
    
    for root, _, files in os.walk(f"./{PRE_DIR}"):
        for file in files:
            old_path = os.path.join(root, file)
            idx[i] = f"{i:06d}.pt"
            new_path = os.path.join(PRE_DIR, idx[i])
            
            if "real" in old_path:
                key[0][i] = 1
            elif "fake" in old_path:
                if "DEV" in old_path:
                    key[1][i] = 1
                elif "PRO" in old_path:
                    key[2][i] = 1
                elif "SDXL" in old_path:
                    key[3][i] = 1
                else:
                    raise Exception("UNEXPECTED DATA FOUND, PLEASE REMOVE DIRECTORY AND RE-PROCESS")
            else:
                raise Exception("UNEXPECTED DATA FOUND, PLEASE REMOVE DIRECTORY AND RE-PROCESS")
        
            shutil.move(old_path, new_path)
            i = i + 1
        
    df = pd.DataFrame.from_dict({
    "FILE": idx.tolist(),
    "REAL": key[0].tolist(),
    "FLUX_DEV": key[1].tolist(),
    "FLUX_PRO": key[2].tolist(),
    "SDXL": key[3].tolist()
    })
    
    for d in FLUX_DIR:
        os.rmdir(f"./{PRE_DIR}/{FAKE_DIR}/{d}")
        
    os.rmdir(f"./{PRE_DIR}/{REAL_DIR}")
    os.rmdir(f"./{PRE_DIR}/{FAKE_DIR}")
    
    df.to_csv(f"{PRE_DIR}/index.csv", index=False)

    print("DONE")
else:
    print("SANITIZED DATA FOUND, CONTINUING")
    
df = pd.read_csv(f"{PRE_DIR}/index.csv")

try:
    assert all(df.drop("FILE", axis=1).sum(axis=0).values == [N_REAL, N_FAKE[0], N_FAKE[1], N_FAKE[2]])
    assert all(df.drop("FILE", axis=1).sum(axis=1).values == 1)
    assert len(os.listdir(f"{PRE_DIR}")) == N + 1
    assert os.path.exists(f"{PRE_DIR}/index.csv")
except:
    raise Exception("DAMAGE TO PREPROCESSED DATA DETECTED, PLEASE REMOVE DIRECTORY AND RE-PROCESS")

## Model training

For this model training, a smaller split is used to enable local training. For final training the full data frame will be used.

In [ ]:
from sklearn.model_selection import train_test_split

# df_local = df.sample(n=800, random_state=SEED)

df_local_train, df_local_validation = train_test_split(df, test_size=0.2, random_state=SEED)


Make class to be used to easily pull information from the data frame

In [ ]:
from torch.utils.data import Dataset , DataLoader 

class data_set_pt(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx] 
        file_path = os.path.join(PRE_DIR,row["FILE"])
        tensor = torch.load(file_path)
        label = torch.tensor([float(row["REAL"]),1.0-float(row["REAL"])])

        if self.transform:
            tensor = self.transform(tensor)

        return tensor, label
    
local_data_set_train = data_set_pt(df_local_train)
local_data_set_validation = data_set_pt(df_local_validation)


In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(
    local_data_set_train,
    batch_size=BATCH_SIZE, ## size held in memory
    shuffle=True
)
validation_loader = DataLoader(
    local_data_set_validation,
    batch_size=BATCH_SIZE, ## size held in memory
    shuffle=True
)


imports and verify use of GPU

In [ ]:
import torchvision.models as models
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 100

print (DEVICE)
print ("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

## Model Training

This only needs to be run when you are training new models, do not run unless you want to train the model


In [ ]:
model = models.efficientnet_b0().to(DEVICE) ## weights and learning rate need to be set

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) 
## stole these two, do not know if they are optimal

classifier = nn.Linear(model.classifier[1].out_features, 2).to(DEVICE)
epoch_log = open("epoch_log","w") 

for i in range(EPOCHS):
    model.train()
    training_loss = []
    for X, y in train_loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        y = y.to(torch.float32)
        optimizer.zero_grad()
        y_hat = classifier(model(X)).to(DEVICE)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()
        training_loss.append(loss.item())
    model.eval()
    validation_loss = []
    with torch.no_grad():
        for X, y in validation_loader:
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            y_hat = classifier(model(X))
            loss = criterion(y_hat, y)
            validation_loss.append(loss.item())
    print(f"EPOCH {i+1}: TRAINING LOSS: {1e4*np.mean(training_loss):0.3f} | VALIDATION LOSS: {1e4*np.mean(validation_loss):0.3f}")
    if (i % 5) == 0:
        torch.save(model,f"model_epoch_{i+1}.pt")
        torch.save(classifier,f"classifier_epoch_{i+1}.pt")
    epoch_log.write(f"EPOCH {i+1}: TRAINING LOSS: {1e4*np.mean(training_loss):0.3f} | VALIDATION LOSS: {1e4*np.mean(validation_loss):0.3f}\n")
epoch_log.close()    

## Model Loading and Validation

This section loads and validates the model

In [ ]:
print(f"cwd is {os.getcwd()}")

# change the file paths if you want to load a different model


negitive_control_model = models.efficientnet_b0().to(DEVICE)
file_name_model = "First_long_local_trained/model_epoch_45.pt"
file_name_classifier ="First_long_local_trained/classifier_epoch_45.pt"

if os.path.exists(file_name_model):
    print ("File path exists")
else:
    print("File not found")

trained_model = torch.load(file_name_model, weights_only=False)
trained_classifier = torch.load(file_name_classifier, weights_only=False)

criterion_post_train = nn.CrossEntropyLoss()

trained_model.eval()
validation_loss = []
validation_loss_control = []
with torch.no_grad():
    for X, y in validation_loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        y_hat = trained_classifier(trained_model(X))
        y_hat_control = trained_classifier(negitive_control_model(X))
        loss = criterion_post_train(y_hat, y)
        loss_control = criterion_post_train(y_hat_control, y)
        validation_loss.append(loss.item())
        validation_loss_control.append(loss_control.item())
print(f"VALIDATION LOSS: {1e4*np.mean(validation_loss):0.3f}")
print(f"CONTROL VALIDATION LOSS: {1e4*np.mean(validation_loss_control):0.3f}")

